# 🎯 WE4 · Notebook 02 — Policy Gradients
## Learning *how to act*, one three-day campaign at a time

> **The story.** You run retention at **Owlinguo**, a free language-learning app that about
> **2 million** people open each month. It makes money the simple way: **one ad before every lesson**.
> So a learner is worth exactly what they study — someone on a daily streak earns you a little every
> day, someone who has stopped opening the app earns you nothing.
>
> And your data says the same thing every time: once a learner's streak breaks, you have about
> **three days** before they are gone for good. So the retention team runs a **three-day win-back
> campaign** on each lapsing learner. Every morning they look at how engaged the learner is and choose
> **one** move: leave them alone, send a nudge from their coach, or blast them with extra ads. Some
> moves earn revenue immediately; some buy the *right* to earn more tomorrow.
>
> The team doesn't want a model. They want **one sheet of paper with three lines** — *learner looks
> like this → do that.* Your job in this notebook is to **learn that sheet from experience**, instead
> of settling it by seniority in a meeting.

That sheet has a name in reinforcement learning: a **policy**. And the method we'll use to learn it
— **REINFORCE**, the original policy-gradient algorithm — is the direct ancestor of how modern LLMs
are fine-tuned with RL.

**How this notebook works**
- Short explanations, then small hands-on tasks marked **🎯** for you to complete.
- **Interactive widgets** to play with each idea *before* the maths shows up.
- The campaign is deliberately tiny — small enough that we can compute **every quantity exactly**
  (the objective, its true gradient, the best possible sheet) and check our sampled estimates against
  the truth. That is a luxury you will never have on a real problem, so we use it hard.
- 💰 **All money is expected ad revenue per learner, in CHF.** A reward of `+6.0` means CHF 6 — small
  on its own, and Owlinguo runs this campaign on a few hundred thousand learners a year.

> 🧭 **Part 1 is a recap** of last lecture's vocabulary — if *state, action, transition, policy,
> reward* are already solid, skim it and start at Part 2 (but do run its code cells: they define the
> campaign).


## 0. Setup

This notebook is **self-contained**: the first cell pulls the exercise files (the `pg_viz.py`
display helpers) directly from the course repository. Run the setup cells below in order.

> 🔑 **While the course repo is private** (testing phase) you need a GitHub access token:
> open the **Secrets** panel (🔑 icon in the left sidebar), add a secret named
> **`GITHUB_TOKEN`**, paste your token, and toggle *Notebook access* on. Once the repo is
> public, no token is needed — the cell clones it directly.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE4-public"
HELPER     = os.path.join("2_pg", "exercise", "pg_viz.py")

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding `2_pg/exercise/` — so imports resolve cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()),
              os.path.dirname(os.path.dirname(os.getcwd())), os.getcwd()]:
    if os.path.exists(os.path.join(_root, HELPER)):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (2_pg/exercise/pg_viz.py). If it is still private, add a "
        "GITHUB_TOKEN secret (see the note above) and re-run this cell.")
sys.path.insert(0, os.path.join(os.getcwd(), "2_pg", "exercise"))   # make the helpers importable
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** All of these are already on Colab; this just pins versions
(and makes the notebook work outside Colab too).

In [ ]:
%pip install -q -r 2_pg/exercise/requirements_pg.txt

**0.3 — Import the libraries.** The diagrams, widgets and quizzes live in **`pg_viz`** so the
teaching cells stay about the *idea*, not about HTML.

In [ ]:
import numpy as np
import torch
import itertools

import importlib
import pg_viz
importlib.reload(pg_viz)   # pick up the latest helpers even if a stale copy was cached

torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)
print("Environment ready ✅  ·  torch", torch.__version__)

---
# Part 1 — The RL vocabulary, on our campaign  *(optional recap)*

> 🧭 **Optional.** This part re-derives the five words from last lecture — **state, action,
> transition, policy, reward** — on the campaign we'll use all notebook long. If those are already
> solid, skim the text but **run the code cells**: they define the world everything else uses.

If reinforcement learning had to fit in one sentence, it would be:

> ### **“Learning how to act, given the state I am in.”**

Everything else is bookkeeping around that sentence. Let's take it apart.

In [ ]:
pg_viz.campaign_overview()

## 1.1 · State — what I bother to know about the learner

The **state** `s` is the description of the situation that the decision is allowed to depend on.

The crucial thing — and it trips up almost everyone at first — is that **a state is not a fact about
the world, it is a choice we make**. A learner has an age, a target language, a device, a reason for
signing up, 14 months of history. We decided that what matters for *this* decision is one thing:

$$s \;\in\; \{\ \text{😴 Cold},\ \ \text{🙂 Warm},\ \ \text{🔥 Hot}\ \}$$

**Three states.** That is the entire description of a learner, as far as our policy is concerned.
A good state is the *smallest* summary that is still enough to act well — small enough to learn
from, rich enough to decide with.

The same sheet is used on every day of the campaign — one rule per engagement level, applied every
morning. That is what makes it a *sheet* and not a schedule.

### 🧠 Quick check — what should “state” mean in Pong?

In [ ]:
pg_viz.mc_quiz("pong_state")

### 🧠 One more pass on the idea

In [ ]:
pg_viz.true_false_quiz("state")

## 1.2 · Action & transition — how acting changes the world

An **action** `a` is a move we can make: `⏸️ Wait`, `🔔 Nudge`, `📺 Ad blast`.

Taking an action does two things: it **books a margin now**, and it **moves the learner to a new
state**. That second part is the **transition function**, and it comes in two flavours:

$$\textbf{deterministic:}\quad s' = T(s,a) \qquad\qquad
\textbf{stochastic:}\quad s' \sim P(\,\cdot \mid s,a)$$

- **Deterministic** — the same action from the same state *always* lands in the same place. Clean,
  and completely unlike a person.
- **Stochastic** — it lands *somewhere*, with probabilities. The coach's nudge arrives; sometimes it
  gets the learner back on a streak, sometimes it is swiped away. **That is the world we model** — flip
  between the two in the widget below.

Here are our rules. `TRANS[s][a]` lists the possible next states with their probabilities;
`REWARD[s][a]` is the **expected** margin booked that day.

In [ ]:
# The campaign's rules are three small tables. They read far better drawn than typed,
# so they live in the helper module — run this to see them.
ENGAGE, ACTIONS, N_DAYS     = pg_viz.ENGAGE, pg_viz.ACTIONS, 3
TRANS, REWARD, START_PROBS  = pg_viz.TRANS, pg_viz.REWARD, pg_viz.START_PROBS

pg_viz.mdp_tables(TRANS, REWARD)

Two things about that first table you will need in a minute: it is indexed
**`TRANS[state][action]`**, and each cell is a small list of *(next state, probability)* pairs.

In [ ]:
print("TRANS[1][1]  — a 🙂 Warm learner (state 1) who gets a 🔔 Nudge (action 1):")
print("   ", TRANS[1][1], "  → 70% of the time they end up in state 2 (🔥 Hot), 30% they stay Warm")
print("REWARD[1][1] =", REWARD[1][1], "  — the ad revenue that move books today")
print("START_PROBS  =", START_PROBS, "  — who enters the campaign (nobody starts 🔥 Hot)")

**The economics in one line:** `🔔 Nudge` **buys engagement** — you spend a little today so the
learner studies more tomorrow, and a learner who studies is a learner watching ads. `📺 Ad blast`
**sells engagement** — you squeeze today's audience for all it is worth, and 80% of the time they get
fed up and stop opening the app.

### 🕹️ Walk one learner through the three days

Pick a move for day 0, and the learner moves. Switch between the **deterministic** world and the
**stochastic** one, and in each of them replay the *same* three moves on a new learner: in one you
get the same story back every time, in the other you do not.

In [ ]:
pg_viz.transition_demo()

### 🎯 Task 1 — implement the transition function

This is the world, in three lines. Given a **state** `s` and an **action** `a`, the function has to
produce the two things an environment owes us:

$$s' \sim P(\,\cdot \mid s, a) \qquad\text{and}\qquad r = R(s, a)$$

The reward is a plain lookup in `REWARD`. The next state is a **draw** from the list of
*(next state, probability)* pairs sitting in `TRANS[s][a]`.

> 💡 **`np.random.choice(values, p=probabilities)`** draws one entry from `values`, respecting the
> given probabilities. That single call *is* the world rolling its die.

In [ ]:
def transition(state, action):
    '''The environment's one-step rule: returns (next state, reward).'''
    reward   = REWARD[state][action]
    outcomes = TRANS[state][action]        # e.g. [(2, 0.7), (1, 0.3)] — (next state, probability)

    next_states = [s for s, p in outcomes]
    probs       = [p for s, p in outcomes]
    next_state  = ???        # 🎯 draw ONE next state, respecting probs (which numpy call does that?)
    return int(next_state), float(reward)

# --- self-check: repeat the same (Warm, Nudge) 4000 times and watch the table's 70/30 appear
draws = [transition(1, 1)[0] for _ in range(4000)]
share_hot = np.mean([d == 2 for d in draws])
print(f"Warm + Nudge landed 🔥 Hot in {share_hot:.1%} of 4000 tries  (the table says 70%)")
assert abs(share_hot - 0.7) < 0.05, "That doesn't match the transition table — check the draw."
print("The reward is NOT random — Warm + Nudge always books", transition(1, 1)[1])

### 🧠 Quick check — transitions

In [ ]:
pg_viz.true_false_quiz("transition")

## 1.3 · Policy — how I choose an action, given a state

The **policy** is the object we are actually learning: the rule that turns a state into an action.
It also comes in two flavours, and this is the distinction that matters most for everything after it.

$$\textbf{deterministic:}\quad a = \pi(s) \qquad\qquad
\textbf{stochastic:}\quad a \sim \pi(a \mid s),\quad \textstyle\sum_a \pi(a\mid s) = 1$$

In [ ]:
pg_viz.policy_types()

### 🎯 Task 2 — run one campaign under a fixed sheet

Before any learning, let's just *watch* a campaign happen. Take the sheet **“nudge anyone who isn't
Hot, cash in when they are”** and run it for three days on one learner.

Each day: read the action off the sheet, apply it, record what happened, and move to tomorrow's
engagement. You already have the one-step rule from Task 1 — this loop just calls it three times.

In [ ]:
#        😴 Cold  🙂 Warm  🔥 Hot
sheet = [1,       1,       2]      # 🔔 Nudge, 🔔 Nudge, 📺 Ad blast

def run_campaign(sheet, start_engagement=None):
    '''Three days under a fixed (deterministic) sheet. Returns states, actions, rewards.'''
    engagement = (int(np.random.choice(len(ENGAGE), p=START_PROBS)) if start_engagement is None
                  else start_engagement)
    states, actions, rewards = [], [], []
    for day in range(N_DAYS):
        action = sheet[engagement]                 # ← the sheet reads ONLY the engagement level
        states.append(engagement); actions.append(action)
        next_engagement, reward = ???              # 🎯 one step of the world — your function from Task 1
        rewards.append(reward)
        engagement = ???                           # 🎯 tomorrow's starting point: which value above?
    return states, actions, rewards

np.random.seed(4)
for run in range(2):                               # SAME sheet, SAME starting state, twice
    s, a, r = run_campaign(sheet, start_engagement=0)
    pg_viz.episode_strip(s, a, r, title=f"Run {run+1} — sheet 🔔/🔔/📺 on a 😴 Cold learner")

**Same sheet, same learner, two different stories.** Nothing about our decision-making changed
between those runs — the world simply rolled differently. Hold on to that: from here on, when we
measure how good a sheet is, **one campaign is never evidence.** We will always be averaging.

That sequence of *(state, action, reward)* triples is called a **trajectory** (or **episode**),
written **τ**. Ours are always three days long.

### 🧠 So why train a *stochastic* policy, if we ship a deterministic sheet?

In [ ]:
pg_viz.mc_quiz("policy_kind")

## 1.4 · Reward & return — what “good” means

A **reward** `r` is the margin booked on a single day. But we don't want to maximise *a day*, we want
to maximise *the campaign*. The standard way to add up an episode is the **discounted return**:

$$G(\tau) \;=\; r_0 \;+\; \gamma\, r_1 \;+\; \gamma^2 r_2 \;+\;\dots\;=\; \sum_{t=0}^{T-1} \gamma^{t}\, r_t
\qquad \gamma \in [0, 1]$$

The **discount factor γ** is the dial for **short-sightedness**:
- **γ = 0** — only today counts. Every future franc is multiplied by zero.
- **γ = 1** — a franc on day 3 is worth exactly a franc today. Perfectly patient.

> 💡 **γ is *our* choice, not the learner's.** It is part of how *we* define success — a business
> decision that we happen to write as a number. Slide it below on a campaign that nudged twice and
> then cashed in on a Hot learner.

In [ ]:
pg_viz.gamma_slider([-0.5, 0.5, 6.0],
                    labels=["day 0 · 😴 → 🔔 Nudge", "day 1 · 🙂 → 🔔 Nudge", "day 2 · 🔥 → 📺 Ad blast"])

### 🎯 Task 3 — the discounted return, in code

Turn that formula into three lines. Walk over the rewards, weight each one by `gamma ** t`, add them
up. (You'll reuse this function in every remaining part of the notebook, so keep it plain.)

In [ ]:
GAMMA = 0.9      # Owlinguo values tomorrow's franc at 90% of today's

def discounted_return(rewards, gamma=GAMMA):
    '''G = r_0 + gamma*r_1 + gamma^2*r_2 + ...'''
    total = 0.0
    for t, r in enumerate(rewards):
        total += ???        # 🎯 this day's reward, weighted by how far in the future it is
    return total

# --- self-check against two γ values you can verify in your head
assert abs(discounted_return([-0.5, 0.5, 6.0], 1.0) - 6.0) < 1e-9, "γ=1 is just the plain sum."
assert abs(discounted_return([-0.5, 0.5, 6.0], 0.0) + 0.5) < 1e-9, "γ=0 keeps only day 0."
print("G of the 🔔🔔📺 campaign at γ = 0.9 :", round(discounted_return([-0.5, 0.5, 6.0]), 3))

### 🔢 Predict before you compute

In [ ]:
pg_viz.number_quiz("returns")

### 🧠 Quick check — the discount factor

In [ ]:
pg_viz.true_false_quiz("gamma")

**Where we stand.** We have a world (`transition`), a way to score a campaign
(`discounted_return`), and a name for what we want (a policy). What we do *not* have is a policy made
of **numbers we can tune**. That's next.

---
# Part 2 — A policy made of numbers

To *learn* a policy we need it to be a **function with parameters** — turn a dial, and the behaviour
changes a little. The standard construction for a handful of discrete actions:

1. keep one raw score, called a **logit**, for each action in each state — that is our parameter `θ`
   (3 states × 3 actions, so **9 numbers** in total);
2. in a given state, push its three numbers — **one per action** — through a **softmax** to turn them
   into three probabilities;
3. **sample** the action from those probabilities.

$$\pi_\theta(a \mid s) \;=\; \operatorname{softmax}\big(\theta_s\big)_a
\;=\; \frac{e^{\theta_{s,a}}}{\sum_{a'} e^{\theta_{s,a'}}}$$

The exponential does two jobs at once: it makes every number **positive**, and dividing by the sum
makes them **add up to 1**. That's all a probability distribution needs.

### 🧭 A detour worth taking — why do it this way at all?

With three actions this feels like bureaucracy: we could keep a table of "best action per state" and
be done. So before we build it, here is why this particular construction — *parameters that describe
a distribution over actions* — is the one the whole field is built on.

In [ ]:
pg_viz.why_parametrise()

For our campaign, honestly, either route works — three actions is a lookup either way. Keep the
right-hand panel in mind anyway: it is the reason the algorithm we are about to write is the same one
that steers robots and fine-tunes language models, while the left-hand one is not.

### 🎯 Task 4 — softmax, by hand

Two lines: exponentiate, then divide by the sum.

> 💡 **`np.exp(x)`** is `e` to the power `x`, element-wise on an array. Its useful property here is
> that it is **positive for every input** — `np.exp(-2)` is small but still above zero — so it turns
> any three real numbers into three positive ones.

In [ ]:
def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    e = ???                 # 🎯 turn each logit into a positive number — the function from the hint
    return ???              # 🎯 turn those positive numbers into something that sums to 1

p = softmax([0.0, 1.0, -1.0])
print("probabilities:", p.round(3), " sum =", round(float(p.sum()), 6))
assert abs(p.sum() - 1) < 1e-9 and (p > 0).all(), "Must be positive and sum to 1."
# equal logits -> a uniform policy, whatever the shared value is
assert np.allclose(softmax([0, 0, 0]), softmax([7, 7, 7])), "Only the GAPS between logits matter."
print("softmax([0,0,0]) =", softmax([0, 0, 0]).round(3), " ·  softmax([7,7,7]) =", softmax([7, 7, 7]).round(3))

### 🕹️ Play with it — three logits, three probabilities

Drag the logits around. Two things are worth *feeling* rather than reading:
- adding the **same** amount to all three logits changes **nothing** (press the button);
- the bigger the **gap**, the more deterministic the policy becomes.

Then hit *Sample 20 actions* a few times: this is what "the policy chooses" actually looks like.

In [ ]:
pg_viz.softmax_playground()

### The full policy: one row of logits per state

Our state is the engagement level, so the parameter is a very small table:

$$\theta \in \mathbb{R}^{\,3\ \text{states}\ \times\ 3\ \text{actions}}
\qquad = \qquad \textbf{9 numbers.}$$

Every sheet we can express — the untrained one, the optimal one, and everything in between — is
*some* setting of those 9 numbers. Starting them all at **zero** gives a uniform policy: every action
equally likely everywhere. That is the right place to start, because it has no opinions yet.

In [ ]:
theta = torch.zeros(len(ENGAGE), len(ACTIONS), requires_grad=True)

def action_probs(theta, engagement):
    '''π(·|s) — the softmax of that state's row of logits.'''
    return torch.softmax(theta[engagement], dim=-1)

print("π(·| 😴 Cold) =", action_probs(theta, 0).detach().numpy().round(3), " → uniform, as promised")

### 🎯 Task 5 — sample a whole campaign from the policy

Now put Parts 1 and 2 together: draw a learner, then three days of **sampling** the action from
`π(·|s)` instead of reading it off a fixed sheet.

Two new pieces. The first is the sampling itself — meet it on its own before we use it:

In [ ]:
# torch.multinomial(p, 1) draws ONE index from a probability vector.
p_demo = torch.tensor([0.1, 0.7, 0.2])          # 10% action 0 · 70% action 1 · 20% action 2
draws  = [int(torch.multinomial(p_demo, 1)) for _ in range(1000)]
print("one draw :", int(torch.multinomial(p_demo, 1)))
print("out of 1000 draws:", [draws.count(a) for a in range(3)], " ≈ 100 / 700 / 200 ✅")

The second is that we keep **`log π(a|s)`** for the action we actually took. Ignore why for
now; it is the single quantity the whole learning algorithm is built on, and Part 3 explains it.

In [ ]:
def sample_episode(theta):
    '''One 3-day campaign on one learner, sampling each action from the policy.
    Returns states, actions, rewards, and the log-probabilities of the chosen actions.'''
    engagement = int(np.random.choice(len(ENGAGE), p=START_PROBS))     # which learner walked in today
    states, actions, rewards, log_probs = [], [], [], []
    for day in range(N_DAYS):
        p = action_probs(theta, engagement)
        action = int(torch.multinomial(???, 1))       # 🎯 draw an action — from which distribution?
        log_probs.append(torch.log(p[???]))           # 🎯 the log-prob of the action we ACTUALLY took
        next_engagement, reward = transition(engagement, action)
        states.append(engagement); actions.append(action); rewards.append(reward)
        engagement = next_engagement
    return states, actions, rewards, log_probs

s, a, r, lp = sample_episode(theta)
print("states :", [ENGAGE[x] for x in s])
print("actions:", [ACTIONS[x] for x in a])
print("rewards:", r, " · G =", round(discounted_return(r), 3))
# under the uniform start policy every action has probability 1/3
assert all(abs(float(torch.exp(l.detach())) - 1/3) < 1e-6 for l in lp), "Uniform policy → log(1/3) every time."
print("log-probs:", [round(float(l.detach()), 3) for l in lp], " = log(1/3) three times ✅")

### Run it a few times — the policy is a *distribution over campaigns*

Same `theta`, eight campaigns. Look at the **G column on the right**: that is what each campaign
earned, and the eight numbers are all over the place. Two things caused that spread — the policy
sampled different actions, and the world answered differently — and neither is a mistake. It is
simply what "how good is this policy?" has to be measured through, and it will come back to haunt us
in Part 4.

In [ ]:
batch = [sample_episode(theta)[:3] for _ in range(8)]
pg_viz.trajectory_gallery(batch, GAMMA)

---
# Part 3 — What exactly are we maximising?

A policy doesn't produce *a* campaign, it produces a **distribution over campaigns**. So "how good is
this sheet" cannot mean the return of one campaign — it has to be the **average return over all the
campaigns it could produce**, weighted by how likely each one is. That is the **objective**:

$$\boxed{\;J(\theta) \;=\; \mathbb{E}_{\tau \sim \pi_\theta}\big[\,G(\tau)\,\big]
\;=\; \sum_{\tau} P(\tau \mid \theta)\; G(\tau)\;}$$

Now, what is `P(τ|θ)`? Two sets of dice get rolled — ours and the world's — so a trajectory's
probability **factorises into exactly those two parts**:

$$P(\tau\mid\theta) \;=\;
\underbrace{\rho(s_0)\prod_{t} P(s_{t+1}\mid s_t, a_t)}_{\textbf{the world's part — fixed}}
\;\times\;
\underbrace{\prod_{t} \pi_\theta(a_t \mid s_t)}_{\textbf{the policy's part — ours to move}}$$

`ρ(s₀)` is just which learner walked in. **Only the right-hand factor contains θ** — that sentence is
what the rest of the notebook is built on.

### First, every story the campaign could tell

Our campaign is small enough to walk the **entire** tree of what could happen. The helper below does
that bookkeeping and hands back one entry per possible trajectory: the states visited, the actions
taken, the rewards, and the **world's part** of its probability.

In [ ]:
ALL_TRAJ = pg_viz.enumerate_trajectories(GAMMA)
pg_viz.trajectory_tree(ALL_TRAJ)

Two of those stories are worth looking at by name — and they are a good check that you are
reading `P(τ|θ)` the way we just wrote it.

In [ ]:
def policy_probability(theta, traj):
    '''The POLICY's part of P(tau|theta): the product over the 3 days of pi(a_t | s_t).'''
    p = 1.0
    for s, a in zip(traj["s"], traj["a"]):
        p *= float(action_probs(theta, s).detach()[a])
    return p

best  = max(ALL_TRAJ, key=lambda t: t["G"])
worst = min(ALL_TRAJ, key=lambda t: t["G"])
for name, t in [("best ", best), ("worst", worst)]:
    print(f"{name}: " + " → ".join(f"{ENGAGE[x]}/{ACTIONS[y]}" for x, y in zip(t["s"], t["a"])))
    print(f"        G = {t['G']:+.2f}   ·   policy's part {policy_probability(theta, t):.4f}"
          f"   ·   world's part {t['p_world']:.4f}")

# a check on the whole enumeration: the probabilities of all the stories must add up to 1
total = sum(policy_probability(theta, t) * t["p_world"] for t in ALL_TRAJ)
print(f"\nSum of P(tau|theta) over all {len(ALL_TRAJ)} trajectories: {total:.6f}  OK")

> ⚠️ **A trajectory is not a strategy.** That "best" line is a *fluke*, not a plan — look at how
> small its world-part is. What we are after is the best **sheet**: the rule that pays best *on
> average*, over everything the world might do. We compute that in a moment.

### 🔢 Three probabilities to check you have it

In [ ]:
pg_viz.number_quiz("probs")

### 🕹️ The whole objective, on one screen

Every trajectory, with **both parts** of its probability, what it earned, and — in the last column —
**how much it pulls the average**: a story that earns a lot but almost never happens moves the average
very little. Add that last column up and you have `J(θ)`.

Slide the logits for a 😴 Cold learner and watch: the purple column moves, the orange one never
does, and `J` follows. **That is the entire game** — we never change what a trajectory *pays*, or how
the world behaves, only how *likely* our own choices are.

In [ ]:
theta_demo = theta.detach().clone()
theta_demo[0] = torch.tensor([0.0, 0.6, -0.3])     # a slightly opinionated policy for Cold customers
pg_viz.trajectory_enumerator(ALL_TRAJ, theta_demo.tolist(), GAMMA, focus_state=0)

In [ ]:
pg_viz.expectation_recap()

### 🎯 Task 6 — compute `J(θ)` exactly

You have all the pieces: for each trajectory, multiply the two parts of its probability by what it
earned, and add it to the running total. This is a luxury function — in any real problem there are far
too many trajectories to enumerate — but here it gives us **ground truth** to check every estimate
against.

One detail in the scaffolding: we accumulate `log_p`, the **log** of the policy's part, because adding
logs is the same as multiplying probabilities and it keeps the numbers well-behaved.

In [ ]:
def exact_J(theta, gamma=GAMMA):
    '''J(θ) = Σ_τ P(τ|θ)·G(τ), summed over every trajectory. Differentiable.'''
    J = torch.zeros(())
    for traj in ALL_TRAJ:
        G = discounted_return(traj["r"], gamma)          # this trajectory's return, at THIS gamma
        log_p = torch.zeros(())
        for s, a in zip(traj["s"], traj["a"]):
            log_p = log_p + torch.log_softmax(theta[s], dim=-1)[a]    # log π(a_t|s_t), summed
        J = J + ???        # 🎯 how much this trajectory pulls the average: BOTH parts of P(τ), times G
                           #    hint: torch.exp(...) undoes the log and gives you back the policy's part;
                           #          the world's part is sitting in traj["p_world"]
    return J

J_uniform = exact_J(theta)
print("J(uniform policy)              =", round(J_uniform.item(), 4))
print("J(the opinionated Cold policy) =", round(exact_J(theta_demo).item(), 4))
plain = float(np.mean([t["G"] for t in ALL_TRAJ]))
print(f"\n(For contrast, the plain average of all {len(ALL_TRAJ)} returns is {plain:.4f} — a different")
print(" number, because trajectories are NOT equally likely: the world weights them too.)")

### The answer, before we start — all 27 sheets

A **sheet** assigns one action to each of the three engagement levels. Three independent choices, three
options each:

$$\underbrace{3}_{\text{😴 Cold}} \times \underbrace{3}_{\text{🙂 Warm}} \times
\underbrace{3}_{\text{🔥 Hot}} \;=\; 3^3 \;=\; \textbf{27 possible sheets.}$$

> 💡 Don't confuse that with the **9 numbers** in `θ`. The 9 are the *logits* — one per (state,
> action) cell — and they let the policy express not just those 27 sheets but every blend in between
> (60/40 between a nudge and an ad blast, say). The 27 are the *corners* of that space: the
> hand-written sheets a person could put on paper. Training moves through the blends and, when it is
> confident, ends up in a corner.

Since we were handed the transition table, we can do the thing the algorithm is *not* allowed to do:
evaluate all 27 exactly and look up the winner. This is our target for the rest of the notebook.

In [ ]:
def evaluate_sheet(sheet, gamma=GAMMA):
    '''Exact J of a deterministic sheet: keep only the trajectories that sheet can produce.'''
    return sum(t["p_world"] * discounted_return(t["r"], gamma)
               for t in ALL_TRAJ
               if all(a == sheet[s] for s, a in zip(t["s"], t["a"])))

ranked = sorted(((evaluate_sheet(list(sh)), list(sh))
                 for sh in itertools.product(range(len(ACTIONS)), repeat=len(ENGAGE))), reverse=True)

print("The 3 best sheets, and the worst one:")
for J, sh in ranked[:3] + ranked[-1:]:
    print(f"   J = {J:+.3f}   " + " · ".join(f"{ENGAGE[s]}→{ACTIONS[a]}" for s, a in enumerate(sh)))

J_BEST = ranked[0][0]
print(f"\nTarget to beat: J = {J_BEST:+.3f}   ·   the uniform policy today: J = {J_uniform.item():+.3f}")

### 🧠 Quick check — the objective

In [ ]:
pg_viz.true_false_quiz("objective")

## 3.5 · From objective to gradient — and the one obstacle

### 1 · This is an optimization problem, upside down

You've met this shape before. Fine-tuning a model *minimises* a loss:

$$\theta^\star = \arg\min_\theta \; \mathbb{E}\big[L\big],
\qquad \theta \leftarrow \theta - \alpha \nabla_\theta \mathbb{E}[L]$$

We want the opposite — the *most* margin — so we **maximise** and walk **uphill**:

$$\theta^\star = \arg\max_\theta \; J(\theta),
\qquad \theta \leftarrow \theta \;{\color{green}+}\; \alpha \nabla_\theta J(\theta)$$

Same machinery, one sign. (In code we usually keep the familiar minimiser and hand it **`−J`** — a
maximisation of `J` *is* a minimisation of `−J`.) Here is a 1-D slice of `J` along one of its 9
directions, so you can see there really is a hill to climb:

In [ ]:
def J_when_logit_is(x):
    t = theta.detach().clone()
    t[0, 1] = x                      # 😴 Cold → 🔔 Nudge
    return float(exact_J(t))

pg_viz.objective_landscape(J_when_logit_is)

### 2 · …but we cannot differentiate our way in through the returns

Write the gradient out and look at where θ actually lives:

$$\nabla_\theta J(\theta) \;=\; \nabla_\theta \sum_{\tau} P(\tau\mid\theta)\, G(\tau)
\;=\; \sum_{\tau} \big(\nabla_\theta P(\tau\mid\theta)\big)\, G(\tau)$$

**`G(τ)` has no θ in it.** A trajectory that nudged twice and cashed in on a Hot learner pays what it
pays; moving a logit does not change that number. θ only sets **how likely each trajectory is** — it
*parametrises the distribution over trajectories*, not the trajectories' values.

That leaves us stuck in a specific way: that last sum is **not an expectation** any more (it's
weighted by `∇P`, not by `P`), so we cannot estimate it by simply running campaigns and averaging.
And in any real problem we cannot enumerate trajectories either.

### 3 · The log-derivative trick

The way out is one identity from calculus, applied to any positive function `P`:

$$\nabla_\theta P \;=\; P \,\cdot\, \nabla_\theta \log P
\qquad\Big(\text{because } \nabla_\theta \log P = \tfrac{\nabla_\theta P}{P}\Big)$$

Substituting it back turns the sum into an expectation again — and that is the whole point:

$$\boxed{\;\nabla_\theta J(\theta) \;=\; \sum_\tau P(\tau\mid\theta)\, G(\tau)\, \nabla_\theta \log P(\tau\mid\theta)
\;=\; \mathbb{E}_{\tau \sim \pi_\theta}\Big[\, G(\tau)\, \nabla_\theta \log P(\tau\mid\theta) \,\Big]\;}$$

An expectation over campaigns **drawn from our own policy** — which we know how to estimate: run
campaigns, average. We take this theorem as given; the proof adds nothing you need today.

### 4 · And the world quietly drops out

Now the payoff of that factorisation. Take the log of `P(τ|θ)` and the product becomes a sum:

$$\log P(\tau\mid\theta) \;=\; \sum_t \log \pi_\theta(a_t \mid s_t)
\;+\; \underbrace{\log \rho(s_0) + \sum_t \log P(s_{t+1}\mid s_t, a_t)}_{\text{the world — no } \theta \text{ in it}}$$

Differentiating kills the second group entirely — as far as θ is concerned, it is a constant:

$$\nabla_\theta \log P(\tau\mid\theta) \;=\; \sum_{t} \nabla_\theta \log \pi_\theta(a_t\mid s_t)$$

**The transition probabilities have vanished from the update.** We never need to know how the learner
works — not the 70%, not the 80%, none of it. Only the actions we took and what we got paid. That is
the formal version of the box back in Part 1.2: it is what it *means* for a method to be
**model-free**, and it is why the same estimator still works when the "world" is a market, a factory,
or a person reading a generated answer.

### 🧠 One question — what did the trick buy us?

In [ ]:
pg_viz.mc_quiz("log_trick")

---
# Part 4 — Reading the policy gradient like a human

Put the two boxes together and the estimator we will actually run is:

$$\nabla_\theta J \;\approx\; \frac{1}{N}\sum_{i=1}^{N} \; G(\tau_i)\, \sum_{t} \nabla_\theta \log \pi_\theta(a^i_t \mid s^i_t)$$

Read it as a sentence, ignoring every symbol:

> **Run some campaigns. For each one, make all the actions it took more likely — in proportion to how
> much margin it made.**

That's it. Good campaigns pull their own actions up; bad campaigns (negative return) push theirs
down. Nobody ever told the algorithm which *action* was good — only which *campaign* was.

Here is that rule applied to eight campaigns the policy might produce — each one's return decides
which way its own actions get pushed, and how hard.

In [ ]:
rng = np.random.default_rng(3)
sample = [ALL_TRAJ[i] for i in rng.choice(len(ALL_TRAJ), size=8, replace=False)]
pg_viz.push_pull_viz(sample)

### ⚓ The anchor — the one thing to hold on to

**A policy gradient has no idea which action was right. It only knows what a whole campaign earned,
and it shifts probability toward whatever the good campaigns happened to do.** Everything anyone
builds on top of this, which you will see later today and tomorrow — actor–critic, PPO, the RL that
tunes LLMs — is that same move, made less noisy.

Three consequences fall straight out of it, and they are worth keeping even if every formula below
fades:

1. the sheet you ship is a **policy**, learned from sampled outcomes rather than from labels;
2. it can only learn about actions it **actually tries** — which is why the training policy samples;
3. it pursues the **objective you wrote down** (`J`, and the γ inside it) exactly and relentlessly.

And one problem, which the next two sections exist to fix: the rule above is *correct*, but as
written it is far too noisy to use.

## 4.1 · Fix #1 — the return-to-go

Start from the estimator we just wrote, and look at the weight each action gets:

$$\nabla_\theta J \;\approx\; \frac{1}{N}\sum_i \; \sum_t \nabla_\theta \log \pi_\theta(a^i_t\mid s^i_t)
\;\cdot\; {\color{#c0554e}G(\tau_i)}$$

Every action in the campaign is multiplied by the **same** number, `G(τ)`. Now write that number out:

$$G(\tau) \;=\; \underbrace{r_0 + \gamma r_1}_{\text{already banked}} \;+\; \gamma^2 r_2
\qquad\text{(for our three-day campaign)}$$

Take the action on **day 2**. Its weight includes `r_0` and `γr_1` — revenue that was booked *before
that action was taken*. Whatever else is true, the day-2 action cannot have caused them. Including
them does not make the action look better or worse *on average*, but it does make its score jump
around for reasons that have nothing to do with it.

So let's give each action only the part of the campaign that came **after** it:

$$G_t \;=\; r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots \;=\; \sum_{t' \ge t} \gamma^{\,t'-t} r_{t'}$$

This is the **return-to-go**: *"from day t onwards, what is this campaign still going to earn?"* With
it, the estimator becomes

$$\nabla_\theta J \;=\; \mathbb{E}\Big[\sum_t \nabla_\theta \log \pi_\theta(a_t\mid s_t)\;\cdot\;
{\color{#1d7a46}\gamma^{t} G_t}\Big]$$

(the `γ^t` in front is just the day-0 value of a franc earned on day `t`). Each action is now scored by
**its own future**, and nothing else — and this is still the same gradient we started with, only
quieter.

### 🎯 Task 7 — returns-to-go

The clean way to compute these is **backwards**: the last day's return-to-go is just its reward, and
every earlier day is *its own reward plus γ times the one after it*.

$$G_t \;=\; r_t \;+\; \gamma\, G_{t+1}$$

In [ ]:
def returns_to_go(rewards, gamma=GAMMA):
    '''G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...  for every t.'''
    out = [0.0] * len(rewards)
    for t in reversed(range(len(rewards))):     # walk the campaign BACKWARDS
        later = out[t + 1] if t + 1 < len(rewards) else 0.0   # what we already computed for tomorrow
        out[t] = ???         # 🎯 today's reward, plus the discounted value of everything after it
    return out

g = returns_to_go([-0.5, 0.5, 6.0])
print("rewards        :", [-0.5, 0.5, 6.0])
print("returns-to-go  :", [round(x, 3) for x in g])
assert abs(g[2] - 6.0) < 1e-9, "On the last day there is no future — G_2 is just r_2."
assert abs(g[0] - discounted_return([-0.5, 0.5, 6.0])) < 1e-9, "G_0 is the full discounted return."
print("\nG_0 is the whole campaign's return, G_2 is only the last day's — each action is now")
print("scored by what happened AFTER it. ✅")

### 🧠 Why is that swap allowed?

In [ ]:
pg_viz.mc_quiz("rtg")

## 4.2 · Fix #2 — compared to *what*? The baseline

One problem left, and it is the one that decides whether this algorithm is usable. To see it we need
the reward table in front of us again:

In [ ]:
pg_viz.reward_table()

Every campaign that reaches 🔥 Hot therefore collects a big positive return, and the update
**pushes up every action it happened to take there** — including the mediocre ones. The good ones are
pushed slightly harder, so the *ranking* is right, but most of the signal is just *"things went
fine"*, repeated loudly, and that part is pure noise.

The fix is the way any manager already reads a number:

> **An outcome is only good or bad *relative to what you normally expect in that situation*.**

Subtract that expectation — the **baseline** `b` — and score each action by the difference, the
**advantage**:

$$A_t \;=\; G_t - b(s_t) \qquad\Longrightarrow\qquad
\nabla_\theta J \;=\; \mathbb{E}\Big[\sum_t \gamma^t\, \big(G_t - b(s_t)\big)\, \nabla_\theta \log \pi_\theta(a_t\mid s_t)\Big]$$

Note what `b` is allowed to look at: **the situation**, `s_t`. A 🔥 Hot learner is normally worth far
more than a 😴 Cold one, so "did this go well?" has to be asked separately for each — one baseline
number per engagement level.

This is **legal for free**: as long as `b` doesn't depend on the *action* taken, its contribution
averages out to exactly zero, so the expected gradient is unchanged. It steers toward the same policy
— it just gets there with far less shouting.

<details>
<summary>📐 <b>Why subtracting it changes nothing on average</b> (optional, three lines)</summary>

The term we are adding to the estimator is `−b(s)·∇log π(a|s)`. Fix a state `s` and average it over
the actions the policy would take there:

$$\mathbb{E}_{a\sim\pi_\theta(\cdot|s)}\big[\,b(s)\,\nabla_\theta \log \pi_\theta(a\mid s)\big]
\;=\; b(s) \sum_a \pi_\theta(a\mid s)\, \frac{\nabla_\theta \pi_\theta(a\mid s)}{\pi_\theta(a\mid s)}
\;=\; b(s)\, \nabla_\theta \underbrace{\sum_a \pi_\theta(a\mid s)}_{=\,1} \;=\; 0$$

`b(s)` came out of the sum because it does not depend on `a` — that is the *whole* requirement. The
probabilities then sum to 1 no matter what θ is, and the gradient of a constant is zero. Note where
this breaks: a baseline that peeked at *which action was taken* could not be pulled out of the sum,
and would quietly bias every update you make.

</details>

So let's choose one. Below are six occasions on which a campaign reached a 🔥 Hot learner, with what
each collected from that point on. Slide `b(🔥 Hot)` and watch the update change its mind about which
of them were good.

In [ ]:
pg_viz.baseline_effect()

> 🔭 **And we could do better still.** One number for "a Hot learner" already lumps two different
> situations together: being Hot on **day 0** is worth far more than being Hot on **day 2**, simply
> because more campaign remains. A baseline indexed by *(day, engagement)* would separate them — and
> notice that this is allowed, even though our *policy* is not allowed to look at the day: the
> baseline picks no action, so it cannot bias anything. We'll stick to one number per engagement level
> because it is easier to hold in your head, and measure what that choice costs us in a moment.

### 📏 So stop guessing and measure it

We just slid `b(🔥 Hot)` around by hand, and the sweet spot was clearly *what a Hot learner normally
earns from here*. That is not something to guess — it is something to **measure**: run campaigns with
the current policy, and average the return-to-go seen from each engagement level. Nothing more
sophisticated than that.

In [ ]:
theta_exp = torch.zeros(len(ENGAGE), len(ACTIONS), requires_grad=True)
theta_exp.data[0] = torch.tensor([0.0, 0.6, -0.3])      # a policy with some opinion

BASELINE = pg_viz.measure_baseline(sample_episode, returns_to_go, theta_exp)

### 🧾 A bookkeeping change that changes nothing — and breaks everything

Does any of this actually matter? Let finance make a perfectly reasonable request. Owlinguo has just
signed a **brand sponsorship**: a flat **CHF 10 per learner per day**, paid whether or not that
learner opens the app. Finance wants it booked to the campaign like everything else.

Think about what that does to the **decision problem**: every trajectory's return goes up by the same
`10·(1 + γ + γ²)`, so the ranking of sheets is untouched, the best sheet is the same sheet, and even
`∇J` is **exactly** unchanged — adding a constant to a function does not move its slope.

Now think about what it does to an estimator **without** a baseline. Suddenly every campaign returns
something like +30, so every campaign looks like a success and every action it took gets pushed up
hard. The real signal — the difference between a good campaign and a bad one — becomes a rounding
error riding on a big number.

We have ground truth here, so there is nothing to argue about. Below: the **exact** gradient of one
component, then the same number estimated from batches of 16 campaigns under three baselines — none,
ours, and the finer *(day, engagement)* one we set aside above.

In [ ]:
# the exact gradient, via autograd through every trajectory — our ground truth
exact_grad, = torch.autograd.grad(exact_J(theta_exp), theta_exp)
WATCH       = (0, 1)      # the component we track: state 😴 Cold, action 🔔 Nudge
SPONSORSHIP = 10.0        # the flat daily amount finance wants booked

b_state     = pg_viz.measure_baseline(sample_episode, returns_to_go, theta_exp, bonus=SPONSORSHIP)
b_day_state = pg_viz.measure_day_baseline(sample_episode, returns_to_go, theta_exp, bonus=SPONSORSHIP)

pg_viz.variance_experiment(sample_episode, returns_to_go, theta_exp, exact_grad[WATCH], GAMMA,
                           baseline_state=b_state, baseline_day_state=b_day_state,
                           bonus=SPONSORSHIP, watch=WATCH)

### 🔬 What just happened — read this slowly, it is the crux

That histogram is easy to nod at and hard to actually understand. Five steps.

**1 · Remember what the number we are estimating *means*.** It is a gradient — and if we zoom in on a
single parameter, a gradient is just a derivative:

$$\frac{\partial J}{\partial \theta_{\text{Cold},\,\text{Nudge}}}
\qquad = \qquad \text{“should the probability of }🔔\text{ Nudge on a }😴\text{ Cold learner go up, or down?”}$$

A big positive value means *push that action's probability up, hard*. A negative one means *push it
down*. That single number is the entire instruction the update carries about that one cell of the
sheet.

**2 · If we could compute it exactly, we would be done.** The exact value has looked at **every**
story the campaign could produce, each weighted by how likely it is. It knows the true ranking of the
actions in that state — it has seen their consequences everywhere.

**3 · But we cannot compute it. We sample it** — from sixteen campaigns. And sixteen campaigns are a
small, lopsided window on the world.

**4 · Here is how that goes wrong.** Suppose the campaigns we happened to draw are ones where the
learner turned 🔥 Hot early. Look at what the raw returns then say:

In [ ]:
pg_viz.why_baseline_helps()

**5 · Sample enough and it corrects itself — but that is the expensive way.** Push ⏸️ Wait up
today, discover over the next twenty batches that it was a mistake, push it back down. The baseline
skips the round trip: it asks *“better than what a Hot learner usually earns?”*, gets **“no”**, and
pushes down immediately — from the very same four campaigns.

### ⚠️ Not just any baseline

The freedom to choose `b` is not permission to pick anything. Subtracting **17** everywhere would also
be unbiased — and useless, because it centres nothing: every advantage would just be shifted the same
way, and the noise we were trying to cancel would still be there. The baseline only earns its keep
when it is close to **what you genuinely expect in that situation**. That is why it is measured, and
that is why it depends on the state.

### 🎓 The analogy worth remembering

We are, in the end, trying to learn a **ranking of the actions** from a small number of noisy
observations. That is the same problem as this one — press the button.

In [ ]:
pg_viz.grading_analogy()

Map it across and the whole of Part 4.2 fits in one line:

| ranking students | our campaign |
|---|---|
| a student's paper | one sampled step |
| the mark on it | the return-to-go `G_t` |
| **which test they sat** | **the state they were in** |
| the test's average mark | the baseline `b(s)` |
| mark − test average | the advantage `A_t` |

With every paper from the whole year, raw marks are fine — that is the *exact* gradient. With one
paper each, raw marks rank the tests, not the students — that is the sampled gradient without a
baseline. Subtracting what the test normally produces costs nothing and fixes it.

**All three are aiming at the same exact gradient** — no baseline biases anything. What changes is
the *spread*, and it changes enormously: booking a constant daily amount made the baseline-free
estimate wildly unreliable, `b(engagement)` pulled a good part of that back, and
`b(day, engagement)` pulled back nearly all of it.

> 👀 **Worth a second look at that measured table.** With the sponsorship booked, 🙂 Warm scores
> *above* 🔥 Hot. Not because Hot learners are worth less — but because in this campaign a learner is
> usually Hot only near the *end*, when there is little future left to collect, while Warm shows up on
> day 0. One number per engagement level cannot tell "Hot with two days to go" from "Hot on the last
> day", so it splits the difference. That is exactly the gap the third row closes.

That is the whole point of a baseline. It doesn't change *where* we are going; it changes how many
campaigns we must run before we can tell which way that is. And note how little it took to break
things — a bookkeeping change that altered **nothing** about the actual decisions.

> 🔭 **Where this goes next.** Look at what the third row is really doing: it is estimating *how much
> a learner in this situation is worth from here*, and using it as the yardstick. Do that properly —
> **learn** that estimate instead of averaging a batch — and it is called a **value function**. A
> policy plus a value function is an **actor–critic**, the subject of the next notebook.

### 🧠 Quick check — returns-to-go and baselines

In [ ]:
pg_viz.true_false_quiz("baseline")

---
# Part 5 — REINFORCE: the loop

Everything is on the table. Two practical questions remain, and the loop answers both:

- **How often do we update θ?** After a **batch of complete campaigns**, and not one moment sooner.
  Look at why: the weight on day 0's action is its return-to-go, and that number is not known until
  the campaign is over. **So there is no learning during the campaign.** A learner can be handled
  wrongly on day 0 and the algorithm cannot react on day 1 — it has to wait for the ending, and for a
  whole batch of endings, before it changes anything. That is a real limitation, not a detail of our
  implementation, and it is the first thing the **next notebook** fixes: an actor–critic *estimates*
  what the rest of the campaign is worth, which lets it learn from a single step without waiting for
  the end.
- **What do we do with every `(s, a, r)` sample?** Each one contributes exactly **one term** to the
  loss. That is not a convention, it falls out of the estimator: the gradient of a campaign is a
  **sum over its days**,

$$\sum_{t} \gamma^{t} A_t\, \nabla_\theta \log \pi_\theta(a_t \mid s_t)
\qquad\text{so one day} \;=\; \text{one term} \;\gamma^{t} A_t \log \pi_\theta(a_t\mid s_t),$$

  and a batch is just the average of those sums. Then we **throw the sample away**: after the update θ
  has moved, and the old campaigns came from a policy that no longer exists.

In [ ]:
pg_viz.reinforce_loop_diagram()

### The loss we hand to the optimizer

Autograd minimises, and we want to maximise, so we feed it the negative:

$$L(\theta) \;=\; -\,\frac{1}{N}\sum_{i=1}^{N}\sum_{t} \gamma^{t}\,
\big(G^i_t - b(s^i_t)\big)\, \log \pi_\theta(a^i_t\mid s^i_t)$$

Two warnings about this expression, both worth remembering:

- **It is not a loss in the usual sense.** Nothing is being predicted, there is no target, and its
  numerical value means nothing. It exists purely so that `backward()` produces the policy gradient.
  It is called a **surrogate** loss for that reason.
- **The advantage is a constant here.** We differentiate only the `log π` factor; the returns are
  numbers we measured, not functions of θ to be differentiated through.

### One helper, given to you — the batch baseline

Just above, the baseline was measured once on its own batch and frozen. Inside the training loop we
can do it cheaper still: each batch of campaigns *is* a fresh sample of "what normally happens", so we
read the baseline straight off the batch we already collected. Same idea as before — one number per
**engagement level**.

In [ ]:
def batch_baseline(batch, all_G):
    '''b[engagement] = the average return-to-go seen in THIS batch, per engagement level.'''
    sums   = np.zeros(len(ENGAGE))
    counts = np.zeros(len(ENGAGE))
    for i, (states, _, _, _) in enumerate(batch):
        for t, s in enumerate(states):
            sums[s] += all_G[i, t]; counts[s] += 1
    # an engagement level nobody visited in this batch: fall back to the batch average
    return np.where(counts > 0, sums / np.maximum(counts, 1), all_G.mean())

### 🎯 Task 8 — write the REINFORCE loop

Two blanks: the advantage, and the accumulation of the surrogate loss. You have written every
ingredient already — this is the assembly.

> 💡 **The bookkeeping, so you don't have to reverse-engineer it.** Inside the loop, `i` indexes the
> campaign and `t` indexes the day. `all_G[i, t]` is the return-to-go of day `t` of campaign `i`;
> `states[t]` and `log_probs[t]` are that same day's engagement level and log-probability. The
> baseline is the one you just built: `baseline[engagement]`.

In [ ]:
def train_policy(gamma=GAMMA, n_iters=200, batch_size=24, lr=0.08, seed=0, log_every=40):
    torch.manual_seed(seed); np.random.seed(seed)
    theta = torch.zeros(len(ENGAGE), len(ACTIONS), requires_grad=True)
    optimizer = torch.optim.Adam([theta], lr=lr)
    history = []

    for it in range(n_iters):
        # 1 · COLLECT — a batch of complete campaigns under the CURRENT policy
        batch = [sample_episode(theta) for _ in range(batch_size)]

        # 2 · SCORE — the return-to-go of every step of every campaign
        all_G = np.array([returns_to_go(rewards, gamma) for _, _, rewards, _ in batch])

        # 3 · CENTRE — what a campaign in this batch normally earns from each situation
        baseline = batch_baseline(batch, all_G)

        # 4 · PUSH — build the surrogate loss, then one gradient step
        loss = torch.zeros(())
        for i, (states, _, _, log_probs) in enumerate(batch):
            for t in range(N_DAYS):
                advantage = all_G[i, t] - ???   # 🎯 minus what we normally expect in that situation
                #                                  (baseline is indexed by engagement level: states[t])
                loss = loss ???                 # 🎯 accumulate −γ^t · advantage · log π(a_t|s_t)
        loss = loss / batch_size

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 5 · REPEAT — the batch is discarded; the next iteration samples fresh campaigns
        history.append(exact_J(theta, gamma).item())
        if log_every and (it + 1) % log_every == 0:
            print(f"  iter {it+1:4d}   J(θ) = {history[-1]:+.3f}")

    return theta.detach(), history

print("Training on γ = 0.9 …")
theta_trained, history = train_policy()

### Did it learn?

We can plot the **exact** objective at every iteration — again, a luxury of a toy world — against the
untrained policy and against the best of the 27 sheets.

In [ ]:
pg_viz.training_curve(history, j_start=J_uniform.item(), j_best=J_BEST)

### 📄 The deliverable

Now read the policy out as the sheet of paper the retention team asked for. We ship the **favourite
action** in each state — the stochastic policy was the *training* device — and we print the
probability next to it, because a 55% preference and a 99% one are very different pieces of advice.

In [ ]:
def policy_table(theta, n_samples=800):
    '''Action probabilities per engagement level, plus how often each level is seen.'''
    probs  = np.array([torch.softmax(theta[s], dim=-1).numpy() for s in range(len(ENGAGE))])
    visits = np.zeros(len(ENGAGE))
    for _ in range(n_samples):
        states, _, _, _ = sample_episode(theta)
        for s in states:
            visits[s] += 1
    return probs, visits / visits.sum()

probs, visits = policy_table(theta_trained)
pg_viz.action_diagram(probs, visits)

sheet_learned = [int(p.argmax()) for p in probs]
print("The learned sheet scores J =", round(evaluate_sheet(sheet_learned), 3),
      " · the best sheet scores J =", round(J_BEST, 3))

**Read it out loud:** *warm them up, then cash in at the top.* A Cold or Warm learner gets a
nudge — you spend a little to move them up a level — and a Hot learner gets the discount, because
that is the moment the −40% buys the biggest conversion. Nobody encoded that reasoning, and nobody told the
algorithm that a nudge lands 70% of the time. It fell out of sampled campaigns and margins.

### 🎯 Task 9 — change what the business cares about

Here is the payoff of having γ as an explicit knob. Suppose the growth team wants **the money now**.
Same world, same campaign — they just refuse to wait for it. That is a smaller γ: at **γ = 0.3**, a
franc earned on the last day counts for only `0.3² = 9%` of a franc earned today.

Retrain the **same** algorithm on the **same** world with that objective, and compare the two
sheets.

> 💡 Nothing changes except the one argument. `train_policy` already takes γ.

In [ ]:
GAMMA_SHORT = 0.3

print(f"Training on γ = {GAMMA_SHORT} …")
theta_short, history_short = train_policy(???)     # 🎯 same function, one different argument

probs_s, visits_s = policy_table(theta_short)
pg_viz.action_diagram(probs_s, visits_s,
                      title=f"The playbook under a SHORT-SIGHTED objective (γ = {GAMMA_SHORT})")

# each sheet, judged by its OWN objective
best_short = max(evaluate_sheet(list(sh), GAMMA_SHORT)
                 for sh in itertools.product(range(len(ACTIONS)), repeat=len(ENGAGE)))
print(f"γ = 0.9  sheet → J = {history[-1]:+.3f}   (best sheet at γ=0.9 : {J_BEST:+.3f})")
print(f"γ = 0.3  sheet → J = {history_short[-1]:+.3f}   (best sheet at γ=0.3 : {best_short:+.3f})")

**Same world, same algorithm, a different company.** The short-sighted objective stops investing
altogether: it blasts ads at anyone still studying and writes off the Cold learners entirely, because
warming somebody up only pays *after* the campaign is over. Look at the last line of that sheet — the
policy has no opinion about 🔥 Hot learners, and says so, because a campaign that never nurtures
anybody never produces one. Neither sheet is "wrong": they are optimal answers to two different
questions, and the question was set by **us**, in one number.

That is the most important managerial takeaway in the notebook: **in RL, the hard part is not the
optimisation, it is specifying what you are optimising.** The algorithm will pursue whatever you
wrote down, exactly and relentlessly — including the parts you didn't mean.

### 🧠 One last idea — why we can't reuse the campaigns

In [ ]:
pg_viz.mc_quiz("onpolicy")

---
# 🎓 Wrap-up

| The idea | In our campaign | In one line |
|---|---|---|
| **State** `s` | the engagement level | what we choose to know before acting |
| **Action** `a` | wait / nudge / ad blast | what we can do about it |
| **Transition** `P(s'\|s,a)` | the `TRANS` table | how the world answers — with dice |
| **Policy** `π_θ(a\|s)` | softmax over 9 logits | the sheet of paper we're learning |
| **Return** `G(τ)` | discounted 3-day margin | how good one campaign was |
| **Objective** `J(θ)` | `𝔼[G(τ)]` over every trajectory | how good the *sheet* is |
| **Policy gradient** | `𝔼[Σ ∇log π · A_t]` | make what worked more likely |
| **Return-to-go** | what a step still earns | score an action by *its* future only |
| **Baseline** | the batch's average | good or bad *compared to normal* |
| **REINFORCE** | collect → score → centre → push | the loop that ties it together |

**The four things worth carrying out of here**

1. **The gradient never sees a "correct action".** It only sees campaigns and margin. Everything the
   sheet learns, it learns by shifting probability toward what happened to work.
2. **It never sees the world's rules either.** The transition probabilities cancelled out of the
   update. We printed them only to grade ourselves — that is what *model-free* buys you, and it is
   why the same estimator survives when the state is a conversation instead of a three-way label.
3. **Variance is the enemy, not bias.** Return-to-go and baselines don't change *where* the algorithm
   is heading — they change whether it can get there in a sane number of campaigns. Nearly every
   later algorithm (actor–critic, GAE, PPO) is another attack on the same problem.
4. **The objective is a management decision.** γ and the reward table encode what the business wants.
   Task 9 changed one number and got a different company strategy out.

### Where this goes next
- **Notebook 03 — actor–critic:** replace our crude baseline with a *learned* value function.
- **Notebook 04 — GAE & PPO:** stop throwing every batch away after one step, safely.
- **Notebook 05 — RL for LLMs:** the exact same estimator, where the "campaign" is a generated
  answer, the "actions" are tokens, and the "reward" comes from a preference model.

---
## 🏁 Final boss — clear the notebook
Everything you just learned, one statement at a time: **state, transitions, policies, γ, the
objective, the log trick, returns-to-go, baselines, the REINFORCE loop.** The rules: **3 lives**,
**10 seconds** per question, and a wrong answer *or* a timeout costs a life. Reach **10 correct** to
pass. Good luck. 🍀

In [ ]:
pg_viz.flash_quiz()